In [1]:
from tensorflow import keras
from keras.applications.efficientnet import EfficientNetB4
from keras.applications.efficientnet import preprocess_input, decode_predictions
import numpy as np


In [ ]:
model = EfficientNetB4(weights='imagenet')



In [4]:
img_path = '/kaggle/input/ant-image/ant.jpeg'
img = keras.utils.load_img(img_path, target_size=(380, 380))
x = keras.utils.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)
# decode the results into a list of tuples (class, description, probability)
# (one such list for each sample in the batch)
print('Predicted:', decode_predictions(preds, top=3)[0])

1/1 [==============================] - 0s 39ms/step
Predicted: [('n02219486', 'ant', 0.76632285), ('n01773549', 'barn_spider', 0.0064740246), ('n02167151', 'ground_beetle', 0.00616324)]


In [5]:
## Fine Tuning the Model for Cancer Detection

In [6]:
from keras.models import Model
from keras.layers import Dense, GlobalAveragePooling2D

In [7]:
# create the base pre-trained model
base_model = EfficientNetB4(weights='imagenet', include_top=False)

71686520/71686520 [==============================] - 0s 0us/step


In [8]:
# add a global spatial average pooling layer
x = base_model.output
x = GlobalAveragePooling2D()(x)
# let's add a fully-connected layer
x = Dense(1024, activation='relu')(x)
# and a logistic layer -- let's say we have 2 classes
predictions = Dense(1, activation='sigmoid')(x)

In [15]:
# this is the model we will train
model = Model(inputs=base_model.input, outputs=predictions)

# first: train only the top layers (which were randomly initialized)
# i.e. freeze all convolutional InceptionV3 layers
for layer in base_model.layers:
    layer.trainable = False

# compile the model (should be done *after* setting layers to non-trainable)
model.compile(optimizer='rmsprop', loss='binary_crossentropy',metrics=['Accuracy','Precision','Recall'])



In [10]:
train_ds = keras.utils.image_dataset_from_directory(
    '/kaggle/input/melanoma-cancer-dataset/train',
    labels='inferred',
    label_mode='int',
    color_mode='rgb',
    batch_size=32,
    image_size=(380, 380),
    shuffle=True,
    seed=1,
    interpolation='bilinear',
)

Found 11879 files belonging to 2 classes.


In [11]:
train_ds.class_names

['Benign', 'Malignant']

In [12]:
for i,j in train_ds.take(1):
    print(j)

tf.Tensor([0 0 0 0 0 0 1 0 0 0 0 1 1 1 0 1 1 1 0 0 1 0 0 0 1 0 0 0 1 1 0 1], shape=(32,), dtype=int32)


In [13]:
val_ds = keras.utils.image_dataset_from_directory(
    '/kaggle/input/melanoma-cancer-dataset/test',
    labels='inferred',
    label_mode='int',
    color_mode='rgb',
    batch_size=32,
    image_size=(380, 380),
    shuffle=True,
    seed=1,
    interpolation='bilinear',
)

Found 2000 files belonging to 2 classes.


In [16]:
history = model.fit(train_ds,epochs=10,validation_data=val_ds)

Epoch 1/10


2024-02-25 04:39:14.894446: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmodel_2/block1b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1708835958.608354     115 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


372/372 [==============================] - 201s 494ms/step - loss: 0.3365 - Accuracy: 0.8597 - precision: 0.8608 - recall: 0.8372 - val_loss: 0.3117 - val_Accuracy: 0.8760 - val_precision: 0.8643 - val_recall: 0.8920
Epoch 2/10
372/372 [==============================] - ETA: 0s - loss: 0.2801 - Accuracy: 0.8820 - precision: 0.8807 - recall: 0.8665

KeyboardInterrupt: 

In [17]:
model.save('cancer.h5')

/opt/conda/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [20]:
img_path = '/kaggle/input/cancer-sample/C0464485-Lentigo_maligna_melanoma_copy.max-600x600.png'
img = keras.utils.load_img(img_path, target_size=(380, 380))
x = keras.utils.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)

1/1 [==============================] - 0s 38ms/step


In [21]:
preds

array([[0.5745162]], dtype=float32)